## Quantization of the original models

### LLaVA-7b

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "8"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "8"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", load_8bit=True, device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=50,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
)[0]
torch.cuda.empty_cache()

In [ ]:
model

### LLaVA layerwise 

#### Original layer 5

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/ShortGPT/prune_log/llava-v1.5-7b_pruned_5_50_samples/pruned_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-5-dist-2.0-l2-0.5-layer--1-short",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
# model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=10,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
torch.cuda.empty_cache()

#### Original layer 10

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/ShortGPT/prune_log/llava-v1.5-7b_pruned_10_50_samples/pruned_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-10-dist-2.0-l2-0.5-layer--1-short",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
torch.cuda.empty_cache()

#### Original layer 15

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/ShortGPT/prune_log/llava-v1.5-7b_pruned_15_50_samples/pruned_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-15-dist-2.0-l2-0.5-layer--1-short",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

#### Original layer 20

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/ShortGPT/prune_log/llava-v1.5-7b_pruned_21_50_samples/pruned_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-21-dist-2.0-l2-0.5-layer--1-short",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

#### int 8

#### quantized layer 5

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/ShortGPT/prune_log/llava-v1.5-7b_pruned_5_50_samples/pruned_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-5-dist-2.0-l2-0.5-layer--1-short",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
torch.cuda.empty_cache()

#### quantized layer 10

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/ShortGPT/prune_log/llava-v1.5-7b_pruned_10_50_samples/pruned_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-10-dist-2.0-l2-0.5-layer--1-short",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

#### quantized layer 15

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/ShortGPT/prune_log/llava-v1.5-7b_pruned_15_50_samples/pruned_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-15-dist-2.0-l2-0.5-layer--1-short",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

#### quantized layer 21

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/ShortGPT/prune_log/llava-v1.5-7b_pruned_21_50_samples/pruned_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-21-dist-2.0-l2-0.5-layer--1-short",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

## Original width 0.2


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/LLM-Pruner/LLMPruner/prune_log/llava-v1.5-7b_0.2_llava/pytorch_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-0.2-dist-2.0-l2-0.5-layer--1",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

### Origianal width 0.4

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/LLM-Pruner/LLMPruner/prune_log/llava-v1.5-7b_0.4_llava/pytorch_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-0.4-dist-2.0-l2-0.5-layer--1",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

### width 0.2 int8

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/LLM-Pruner/LLMPruner/prune_log/llava-v1.5-7b_0.2_llava/pytorch_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-0.2-dist-2.0-l2-0.5-layer--1",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

### width 0.4 int8

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("${REPO_ROOT}/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '${REPO_ROOT}/LLM-Pruner/LLMPruner/prune_log/llava-v1.5-7b_0.4_llava/pytorch_model.bin',lora="${REPO_ROOT}/VLM/llava/checkpoints/llava-v1.5-7b-0.4-dist-2.0-l2-0.5-layer--1",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('${LLAVA_DATA_ROOT}/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


In [ ]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]